In [1]:
import os
import zstandard  # pip install zstandard
from tqdm import tqdm
import random
import json
import langid
from typing import Generator, Optional, Set

files_processed_to_text = True




In [2]:
from datetime import datetime
def showTime():
    return str("["+datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')+" UTC]")

In [3]:
def zst_files_in_dir(directory):
    """List all .zst files in a directory."""
    files = []
    for filename in os.listdir(directory):
        if filename.endswith(".zst") and os.path.isfile(os.path.join(directory, filename)):
            files.append(filename)
    return files

In [4]:
def decompress_zst_to_text(
    input_file: str,
    vocab: Optional[Set[str]] = None,
    mode: str = "accuracy",
    ascii_threshold: float = 0.5
) -> Generator[str, None, None]:
    """
    Decompresses a .zst file containing JSONL (JSON lines) format, 
    and yields English texts filtered via language detection or ASCII checks.
    
    ### Parameters
    input_file (str):
        Path to the .zst file containing JSONL-formatted lines.
        
    vocab (Optional[Set[str]]):
        A vocabulary set to collect unique characters from valid English texts. Defaults to None.
        
    mode (str, default='accuracy'):
        - 'accuracy': Uses the `langid` library for precise English language detection.
        - 'speed': Uses an ASCII ratio check for faster filtering.
        
    ascii_threshold (float, default=0.5):
        Minimum ASCII character ratio (0.0-1.0) for mode='speed' to consider text as valid.
    
    ### Yields
    str:
        Filtered English text entries from the compressed file.
    """
    with open(input_file, "rb") as infile:
        dctx = zstandard.ZstdDecompressor()
        with dctx.stream_reader(infile) as reader:
            current_line = ""
            while True:
                chunk = reader.read(16384).decode("utf-8", errors="replace")  # Read in 16KB chunks
                if not chunk:
                    break
                current_line += chunk
                # Split into lines (handles partial lines)
                lines = current_line.split("\n")
                current_line = lines.pop() if lines else ""  # Save partial line for next iteration

                # Process each line
                for line in lines:
                    # Skip empty lines after stripping
                    stripped_line = line.strip()
                    if not stripped_line:
                        continue
                    
                    try:
                        data = json.loads(stripped_line)
                        text = data.get("text", "").strip()
                    except (json.JSONDecodeError, KeyError):
                        continue  # Skip invalid JSON
                        
                    except Exception as e:
                        print(f"JSON Error: {e} on line: {line[:50]}...")
                        continue

                    if mode == "accuracy":      
                        # Check language (English)
                        try:
                            detected_lang, _ = langid.classify(text)
                        except langid.langid.LanguageIdentificationError:
                            # Skip texts too short to identify
                            continue

                        if detected_lang != "en":
                            continue  # Non-English, skip

                    elif mode == "speed":
                        # ---- START FILTERING LOGIC ----
                        ascii_count = 0
                        total_chars = 0
                        
                        # Iterate through each character in text
                        for c in text:
                            code = ord(c)
                            if code <= 127:
                                ascii_count += 1
                            total_chars += 1

                        # Check filtering conditions
                        if total_chars == 0:
                            continue
                        if (ascii_count / total_chars) < ascii_threshold:
                            continue
                        # ---- END FILTERING LOGIC ----

                    # Update the vocabulary (only for English texts)
                    if vocab is not None:
                        vocab.update(set(text))

                    yield text.strip()


In [5]:
folder_path = "openwebtext2"
output_file = "output_v7_accuracy.txt"
vocab_file = "vocab_v7_accuracy.txt"


In [6]:
# Gather files
files = zst_files_in_dir(folder_path)
total_files = len(files)
print(f"Total files: {total_files}")
print(files)
vocab = set()

Total files: 179
['2005-06.jsonl.zst', '2005-07.jsonl.zst', '2005-08.jsonl.zst', '2005-09.jsonl.zst', '2005-10.jsonl.zst', '2005-11.jsonl.zst', '2005-12.jsonl.zst', '2006-01.jsonl.zst', '2006-02.jsonl.zst', '2006-03.jsonl.zst', '2006-04.jsonl.zst', '2006-05.jsonl.zst', '2006-06.jsonl.zst', '2006-07.jsonl.zst', '2006-08.jsonl.zst', '2006-09.jsonl.zst', '2006-10.jsonl.zst', '2006-11.jsonl.zst', '2006-12.jsonl.zst', '2007-01.jsonl.zst', '2007-02.jsonl.zst', '2007-03.jsonl.zst', '2007-04.jsonl.zst', '2007-05.jsonl.zst', '2007-06.jsonl.zst', '2007-07.jsonl.zst', '2007-08.jsonl.zst', '2007-09.jsonl.zst', '2007-10.jsonl.zst', '2007-11.jsonl.zst', '2007-12.jsonl.zst', '2008-01.jsonl.zst', '2008-02.jsonl.zst', '2008-03.jsonl.zst', '2008-04.jsonl.zst', '2008-05.jsonl.zst', '2008-06.jsonl.zst', '2008-07.jsonl.zst', '2008-08.jsonl.zst', '2008-09.jsonl.zst', '2008-10.jsonl.zst', '2008-11.jsonl.zst', '2008-12.jsonl.zst', '2009-01.jsonl.zst', '2009-02.jsonl.zst', '2009-03.jsonl.zst', '2009-04.jsonl.z

In [7]:
# Shuffle files randomly 
random.seed(42)  # Optional: Set seed for reproducibility
random.shuffle(files)  # Shuffle in-place
print(files)

['2018-02.jsonl.zst', '2009-11.jsonl.zst', '2006-09.jsonl.zst', '2008-12.jsonl.zst', '2006-07.jsonl.zst', '2008-06.jsonl.zst', '2010-06.jsonl.zst', '2015-08.jsonl.zst', '2010-07.jsonl.zst', '2007-01.jsonl.zst', '2010-11.jsonl.zst', '2007-12.jsonl.zst', '2018-05.jsonl.zst', '2005-08.jsonl.zst', '2016-08.jsonl.zst', '2016-09.jsonl.zst', '2015-06.jsonl.zst', '2018-09.jsonl.zst', '2019-07.jsonl.zst', '2010-12.jsonl.zst', '2013-04.jsonl.zst', '2019-11.jsonl.zst', '2007-07.jsonl.zst', '2016-06.jsonl.zst', '2017-10.jsonl.zst', '2018-08.jsonl.zst', '2019-12.jsonl.zst', '2011-08.jsonl.zst', '2017-03.jsonl.zst', '2017-07.jsonl.zst', '2006-08.jsonl.zst', '2009-10.jsonl.zst', '2016-02.jsonl.zst', '2014-02.jsonl.zst', '2009-02.jsonl.zst', '2015-09.jsonl.zst', '2011-03.jsonl.zst', '2012-02.jsonl.zst', '2018-10.jsonl.zst', '2005-06.jsonl.zst', '2015-01.jsonl.zst', '2006-10.jsonl.zst', '2016-10.jsonl.zst', '2015-11.jsonl.zst', '2013-10.jsonl.zst', '2010-10.jsonl.zst', '2018-06.jsonl.zst', '2012-05.jso

In [8]:
# Process all files
if files_processed_to_text == False:
    with open(output_file, "w", encoding="utf-8") as outf:
        for filename in tqdm(files, total=len(files), desc="Processing Files"):
            print(f"{showTime()} Processing: {filename}")
            file_path = os.path.join(folder_path, filename)
            try:
                for text_line in decompress_zst_to_text(file_path, vocab, mode="accuracy"):
                    outf.write(text_line.strip())  # Write only the text line
            except Exception as e:
                print(f"Error processing {file_path}: {e}")

In [9]:
# Write vocabulary
if files_processed_to_text == False:
    with open(vocab_file, "w", encoding="utf-8") as vfile:
        for char in sorted(vocab):
            vfile.write(char + "\n")

In [10]:
# create instance of BPE tokenizer from tiktoken
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")


In [11]:
encoded_text = tokenizer.encode("Hello, world! I like apple juice - I drink it every day. Isn't that too much?")
print(encoded_text)

[15496, 11, 995, 0, 314, 588, 17180, 13135, 532, 314, 4144, 340, 790, 1110, 13, 25110, 470, 326, 1165, 881, 30]


In [12]:
decoded_text = tokenizer.decode(encoded_text)
print(decoded_text)

Hello, world! I like apple juice - I drink it every day. Isn't that too much?


Encoding the sequence of text

In [13]:
import os
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [14]:
"""
Parallel GPT-2 Tokenization Pipeline with Streaming I/O and Chunked Processing

This script reads a large text file in fixed-size byte batches, safely splits
at word boundaries, tokenizes each chunk using the fast `tiktoken` library,
and writes out intermediate NumPy files as soon as each chunk is done. Finally,
it merges all chunk files into one output array in the correct numerical order.

Key features:
- Streaming file read with “safe” splits to avoid breaking words in half.
- Prefetched batch dispatch to keep CPU cores busy.
- Immediate disk write per chunk to minimize memory pressure.
- Numeric sorting of chunk files to preserve original order.
"""

import os
import re
import logging
import gc
import numpy as np
from time import time
from joblib import Parallel, delayed, parallel_backend
import tiktoken
import importlib

# -- Logging Configuration ----------------------------------------------
# Configure a logger to show timestamped debug/info messages.
logging.basicConfig(
    level=logging.DEBUG,  # Use INFO or WARNING to reduce verbosity in production
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)

def show_time() -> str:
    """
    Return the current time as a formatted string for logging.
    """
    return f"[{time():.2f}]"

# -- Configuration ------------------------------------------------------
FILE_PATH       = "output_v7_accuracy.txt"            # Input text file to encode
BATCH_SIZE      = 100_000_000                         # Read this many bytes at a time
ENC_DIR         = os.path.join("output_v10", "encoded_data")  
OUTPUT_FILENAME = "encoded_output_v10_accuracy.npy"    # Final merged output file
# Number of parallel jobs: half the CPU cores, but at least 1 and no more than 8
n_jobs          = min(max(os.cpu_count() // 2, 1), 8)
# Prefetch factor: read this many batches before dispatching to workers
PREFETCH_FACTOR = 8 # decrease if you run out of memory

# Ensure the output directory exists
os.makedirs(ENC_DIR, exist_ok=True)
logger.info(
    f"Config loaded: FILE_PATH={FILE_PATH}, BATCH_SIZE={BATCH_SIZE}, "
    f"TOKENIZER_MODEL={importlib.metadata.version('tiktoken')}, "
    f"ENC_DIR={ENC_DIR}, n_jobs={n_jobs}"
)

# -- Load Tokenizer ------------------------------------------------------
logger.debug("Initializing tiktoken GPT-2 tokenizer...")
# Uses OpenAI tiktoken for very fast BPE tokenization
tokenizer = tiktoken.get_encoding("gpt2")


def chunk_reader(fp, size):
    """
    Generator that reads from a binary file in byte-sized chunks, decodes to UTF-8,
    and yields text segments that end on a whitespace boundary so words are not split
    across chunks.

    Args:
        fp: Open file object in binary mode.
        size: Number of bytes to read per iteration.

    Yields:
        A string chunk safe to feed into the tokenizer.
    """
    leftover = ""       # Remainder from the previous read to prepend
    total_read = 0      # Total bytes read so far (for logging)
    chunk_index = 0     # Sequential index of each chunk

    while True:
        logger.debug(f"{show_time()} Reading up to {size} bytes from file...")
        data = fp.read(size)
        read_len = len(data)
        total_read += read_len
        logger.info(
            f"{show_time()} Read chunk #{chunk_index} ({read_len} bytes, "
            f"total read {total_read} bytes)"
        )

        # If no more data and no leftover, end iteration
        if not data:
            if leftover:
                logger.debug(f"{show_time()} Yielding leftover ({len(leftover)} chars)")
                yield leftover
            logger.info(f"{show_time()} EOF after {total_read} bytes")
            break

        # Decode bytes to text, prepend leftover from previous chunk
        text = leftover + data.decode("utf-8", errors="ignore")

        # Find last whitespace to avoid splitting a word:
        last_ws = text.rstrip().rfind(" ")
        split_at = max(last_ws + 1, 0)

        if split_at <= 0:
            # No safe split found: yield entire text and clear leftover
            yield text
            leftover = ""
        else:
            # Split into “safe” piece and leftover
            safe = text[:split_at]
            leftover = text[split_at:]
            yield safe

        chunk_index += 1


def tokenize_and_save(text_chunk: str, chunk_id: int):
    """
    Tokenizes a text chunk and immediately saves its token IDs to disk.

    Args:
        text_chunk: The input string to tokenize.
        chunk_id: Sequential ID used to name the output file.

    Returns:
        Tuple of (path_to_saved_file, number_of_tokens).
    """
    pid = os.getpid()
    logger.debug(f"{show_time()} PID {pid} processing chunk #{chunk_id}")

    # Encode the chunk, allowing only the end-of-text special token
    tokens = tokenizer.encode(
        text_chunk,
        allowed_special={'<|endoftext|>'},
        disallowed_special=()
    )
    token_count = len(tokens)

    # Save tokens as a NumPy array for efficient on-disk storage
    temp_path = os.path.join(ENC_DIR, f"chunk_{chunk_id:05d}.npy")
    np.save(temp_path, np.array(tokens, dtype=np.int64))

    logger.info(f"{show_time()} PID {pid} saved {token_count} tokens -> {temp_path}")
    return temp_path, token_count


def run_parallel_encoding():
    """
    Main pipeline:
    1. Reads chunks in a prefetch loop to keep disk I/O busy.
    2. Dispatches chunks in parallel to worker processes.
    3. Collects intermediate .npy files and merges them in order.
    """
    logger.info("🚀 Starting parallel encoding pipeline (streamed per-chunk)")

    total_tokens = 0    # Running count of all tokens encoded
    chunk_id = 0        # Unique ID for each chunk
    temp_files = []     # List of all intermediate file paths

    # Open the input file once
    with open(FILE_PATH, "rb") as f:
        reader = chunk_reader(f, BATCH_SIZE)

        # Loop until no chunks remain
        while True:
            # Prefetch multiple chunks to fill the worker queue
            chunks = []
            for _ in range(n_jobs * PREFETCH_FACTOR):
                try:
                    chunk = next(reader)
                    chunks.append((chunk, chunk_id))
                    chunk_id += 1
                except StopIteration:
                    break

            if not chunks:
                # No more data left
                break

            logger.info(f"{show_time()} Dispatching {len(chunks)} chunks to workers...")

            # Parallel dispatch using loky backend for CPU-bound tasks
            with parallel_backend("loky"):
                results = Parallel(
                    n_jobs=n_jobs,
                    backend="loky",
                    prefer="processes",
                    verbose=5
                )(
                    delayed(tokenize_and_save)(chunk, cid)
                    for chunk, cid in chunks
                )

            # Collect results as soon as each worker finishes
            for temp_path, count in results:
                temp_files.append(temp_path)
                total_tokens += count

            # Occasionally force garbage collection to free memory
            if chunk_id % n_jobs == 0:
                gc.collect()

    # -- Merge Intermediate Files -----------------------------------------
    output_path = os.path.join(ENC_DIR, OUTPUT_FILENAME)
    logger.info(
        f"{show_time()} Merging {len(temp_files)} chunks into final file: {output_path}"
    )

    # Sort by numeric chunk ID to preserve document order
    temp_files.sort(
        key=lambda f: int(re.sub(r'\D', '', os.path.basename(f)))
    )

    # Write all token arrays into one big file
    with open(output_path, "wb") as out_f:
        for tmp in temp_files:
            with open(tmp, "rb") as in_f:
                out_f.write(in_f.read())
            os.remove(tmp)  # Clean up intermediate
            logger.debug(f"{show_time()} Merged & deleted {tmp}")

    # Final log summary
    logger.info(f"{show_time()}  Done. Total tokens: {total_tokens}")
    logger.info(f"Final output: {output_path}")


# -- Entry Point --------------------------------------------------------
if __name__ == "__main__":
    run_parallel_encoding()


14:46:42 [INFO] Config loaded: FILE_PATH=output_v7_accuracy.txt, BATCH_SIZE=100000000, TOKENIZER_MODEL=0.9.0, ENC_DIR=output_v10\encoded_data, n_jobs=8
14:46:42 [DEBUG] Initializing tiktoken GPT-2 tokenizer...
14:46:42 [INFO] 🚀 Starting parallel encoding pipeline (streamed per-chunk)
14:46:42 [DEBUG] [1748090802.01] Reading up to 100000000 bytes from file...
14:46:42 [INFO] [1748090802.47] Read chunk #0 (100000000 bytes, total read 100000000 bytes)
14:46:42 [DEBUG] [1748090802.62] Reading up to 100000000 bytes from file...
14:46:43 [INFO] [1748090803.14] Read chunk #1 (100000000 bytes, total read 200000000 bytes)
14:46:43 [DEBUG] [1748090803.35] Reading up to 100000000 bytes from file...
14:46:43 [INFO] [1748090803.83] Read chunk #2 (100000000 bytes, total read 300000000 bytes)
14:46:44 [DEBUG] [1748090804.05] Reading up to 100000000 bytes from file...
14:46:44 [INFO] [1748090804.54] Read chunk #3 (100000000 bytes, total read 400000000 bytes)
14:46:44 [DEBUG] [1748090804.70] Reading up